# ML용 신규 테이블 EDA — hh_demographic, causal_data, coupon_redempt, product

**목적**: "상위20% 위축 예측 모델"에 쓸 신규 테이블들의 모양·결측·중복을 확인.
(transaction_data는 이미 이전 노트북들에서 충분히 봤으므로 여기선 새로 쓰는 4개 테이블만)

**주의**: feature 계산 시 전부 **초반 구간(week 17~26)** 정보만 사용해야 하므로(데이터 누수 방지),
이 노트북에서도 가능한 부분은 초반 구간으로 필터링한 버전을 같이 확인한다.


In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

candidates = ["Malgun Gothic", "NanumGothic", "AppleGothic"]
available = {f.name for f in fm.fontManager.ttflist}
font_name = next((c for c in candidates if c in available), None)
if font_name:
    plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)

DATA_DIR = "data/"

FILES = {
    "hh_demographic": "hh_demographic.csv",
    "causal_data":    "causal_data.csv",
    "coupon_redempt": "coupon_redempt.csv",
    "product":        "product.csv",
}

dfs = {name: pd.read_csv(DATA_DIR + fname) for name, fname in FILES.items()}

for name, df in dfs.items():
    print(f"{name}: {df.shape}")


hh_demographic: (801, 8)
causal_data: (36786524, 5)
coupon_redempt: (2318, 4)
product: (92353, 7)


## 1. 테이블별 상세 확인 (shape, dtypes, 결측, 중복, 범주값)

In [19]:
def inspect(name):
    df = dfs[name]
    print(f"===== {name}  (shape: {df.shape}) =====\n")

    print("--- head(3) ---")
    display(df.head(3))

    print("\n--- dtypes ---")
    print(df.dtypes)

    n_dup = df.duplicated().sum()
    print(f"\n--- 완전 중복행: {n_dup:,} ({n_dup/len(df)*100:.3f}%) ---")

    print("\n--- 결측치 비율(%) ---")
    na_ratio = (df.isna().sum() / len(df) * 100).round(2)
    print(na_ratio[na_ratio > 0] if na_ratio.sum() > 0 else "결측 없음")

    num_cols = df.select_dtypes(include=np.number).columns
    if len(num_cols) > 0:
        print("\n--- 수치형 describe ---")
        display(df[num_cols].describe().T)

    cat_cols = df.select_dtypes(include='object').columns
    for c in cat_cols:
        nunique = df[c].nunique()
        print(f"\n--- '{c}' 고유값 {nunique}개, 상위 10개 ---")
        print(df[c].value_counts().head(10))

    print("\n")


In [20]:
inspect("hh_demographic")

===== hh_demographic  (shape: (801, 8)) =====

--- head(3) ---


,AGE_DESC,MARITAL_STATUS_CODE,INCOME_DESC,HOMEOWNER_DESC,HH_COMP_DESC,HOUSEHOLD_SIZE_DESC,KID_CATEGORY_DESC,household_key
0,65+,A,35-49K,Homeowner,2 Adults No Kids,2,None/Unknown,1
1,45-54,A,50-74K,Homeowner,2 Adults No Kids,2,None/Unknown,7
2,25-34,U,25-34K,Unknown,2 Adults Kids,3,1,8



--- dtypes ---
AGE_DESC                 str
MARITAL_STATUS_CODE      str
INCOME_DESC              str
HOMEOWNER_DESC           str
HH_COMP_DESC             str
HOUSEHOLD_SIZE_DESC      str
KID_CATEGORY_DESC        str
household_key          int64
dtype: object

--- 완전 중복행: 0 (0.000%) ---

--- 결측치 비율(%) ---
결측 없음

--- 수치형 describe ---


,count,mean,std,min,25%,50%,75%,max
household_key,801.0,1235.17603,736.804647,1.0,596.0,1218.0,1914.0,2499.0



--- 'AGE_DESC' 고유값 6개, 상위 10개 ---
AGE_DESC
45-54    288
35-44    194
25-34    142
65+       72
55-64     59
19-24     46
Name: count, dtype: int64

--- 'MARITAL_STATUS_CODE' 고유값 3개, 상위 10개 ---
MARITAL_STATUS_CODE
U    344
A    340
B    117
Name: count, dtype: int64

--- 'INCOME_DESC' 고유값 12개, 상위 10개 ---
INCOME_DESC
50-74K       192
35-49K       172
75-99K        96
25-34K        77
15-24K        74
Under 15K     61
125-149K      38
100-124K      34
150-174K      30
250K+         11
Name: count, dtype: int64

--- 'HOMEOWNER_DESC' 고유값 5개, 상위 10개 ---
HOMEOWNER_DESC
Homeowner          504
Unknown            233
Renter              42
Probable Renter     11
Probable Owner      11
Name: count, dtype: int64

--- 'HH_COMP_DESC' 고유값 6개, 상위 10개 ---
HH_COMP_DESC
2 Adults No Kids    255
2 Adults Kids       187
Single Female       144
Single Male          95
Unknown              73
1 Adult Kids         47
Name: count, dtype: int64

--- 'HOUSEHOLD_SIZE_DESC' 고유값 5개, 상위 10개 ---
HOUSEHOLD_SIZE_DESC
2

C:\Users\spide\AppData\Local\Temp\ipykernel_21052\2817218715.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns


In [21]:
inspect("causal_data")

===== causal_data  (shape: (36786524, 5)) =====

--- head(3) ---


,PRODUCT_ID,STORE_ID,WEEK_NO,display,mailer
0,26190,286,70,0,A
1,26190,288,70,0,A
2,26190,289,70,0,A



--- dtypes ---
PRODUCT_ID    int64
STORE_ID      int64
WEEK_NO       int64
display         str
mailer          str
dtype: object

--- 완전 중복행: 0 (0.000%) ---

--- 결측치 비율(%) ---
결측 없음

--- 수치형 describe ---


,count,mean,std,min,25%,50%,75%,max
PRODUCT_ID,36786524.0,3.512237e+06,4.094819e+06,26190.0,928850.0,1312148.0,5730125.0,18244453.0
STORE_ID,36786524.0,3.234151e+03,9.106904e+03,286.0,329.0,369.0,421.0,34280.0
WEEK_NO,36786524.0,5.529931e+01,2.696101e+01,9.0,32.0,56.0,78.0,101.0


C:\Users\spide\AppData\Local\Temp\ipykernel_21052\2817218715.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns



--- 'display' 고유값 10개, 상위 10개 ---
display
0    21038745
9     2699467
5     2575289
7     2362118
3     2073738
6     1816021
2     1812840
1     1102141
A      713180
4      592985
Name: count, dtype: int64

--- 'mailer' 고유값 11개, 상위 10개 ---
mailer
A    17106789
0    11534183
D     4467453
H     1560395
F     1077549
J      306924
L      301327
C      291059
X      120823
Z       19453
Name: count, dtype: int64




In [22]:
inspect("coupon_redempt")

===== coupon_redempt  (shape: (2318, 4)) =====

--- head(3) ---


,household_key,DAY,COUPON_UPC,CAMPAIGN
0,1,421,10000085364,8
1,1,421,51700010076,8
2,1,427,54200000033,8



--- dtypes ---
household_key    int64
DAY              int64
COUPON_UPC       int64
CAMPAIGN         int64
dtype: object

--- 완전 중복행: 0 (0.000%) ---

--- 결측치 비율(%) ---
결측 없음

--- 수치형 describe ---


,count,mean,std,min,25%,50%,75%,max
household_key,2318.0,1.302817e+03,7.830025e+02,1.000000e+00,5.880000e+02,1.396500e+03,2.004000e+03,2.500000e+03
DAY,2318.0,5.282174e+02,1.003610e+02,2.250000e+02,4.582500e+02,5.320000e+02,6.050000e+02,7.040000e+02
COUPON_UPC,2318.0,4.123049e+10,1.986068e+10,1.000009e+10,1.000009e+10,5.234003e+10,5.430002e+10,5.897850e+10
CAMPAIGN,2318.0,1.555134e+01,5.716636e+00,1.000000e+00,1.300000e+01,1.400000e+01,1.800000e+01,3.000000e+01


In [23]:
inspect("product")

===== product  (shape: (92353, 7)) =====

--- head(3) ---


,PRODUCT_ID,MANUFACTURER,DEPARTMENT,BRAND,COMMODITY_DESC,SUB_COMMODITY_DESC,CURR_SIZE_OF_PRODUCT
0,25671,2,GROCERY,National,FRZN ICE,ICE - CRUSHED/CUBED,22 LB
1,26081,2,MISC. TRANS.,National,NO COMMODITY DESCRIPTION,NO SUBCOMMODITY DESCRIPTION,
2,26093,69,PASTRY,Private,BREAD,BREAD:ITALIAN/FRENCH,



--- dtypes ---
PRODUCT_ID              int64
MANUFACTURER            int64
DEPARTMENT                str
BRAND                     str
COMMODITY_DESC            str
SUB_COMMODITY_DESC        str
CURR_SIZE_OF_PRODUCT      str
dtype: object

--- 완전 중복행: 0 (0.000%) ---

--- 결측치 비율(%) ---
결측 없음

--- 수치형 describe ---


,count,mean,std,min,25%,50%,75%,max
PRODUCT_ID,92353.0,5.328353e+06,5.359937e+06,25671.0,970628.0,1621091.0,9704770.0,18316298.0
MANUFACTURER,92353.0,1.739228e+03,1.818270e+03,1.0,328.0,1094.0,2264.0,6477.0



--- 'DEPARTMENT' 고유값 44개, 상위 10개 ---
DEPARTMENT
GROCERY       39021
DRUG GM       31529
PRODUCE        3118
COSMETICS      3011
NUTRITION      2914
MEAT           2544
MEAT-PCKGD     2427
DELI           2354
PASTRY         2149
FLORAL          938
Name: count, dtype: int64

--- 'BRAND' 고유값 2개, 상위 10개 ---
BRAND
National    78537
Private     13816
Name: count, dtype: int64

--- 'COMMODITY_DESC' 고유값 308개, 상위 10개 ---
COMMODITY_DESC
GREETING CARDS/WRAP/PARTY SPLY    2785
CANDY - PACKAGED                  2475
MAKEUP AND TREATMENT              2467
HAIR CARE PRODUCTS                1744
SOFT DRINKS                       1704
BAG SNACKS                        1523
HISPANIC                          1460
FRZN MEAT/MEAT DINNERS            1268
STATIONERY & SCHOOL SUPPLIES      1261
MAGAZINE                          1224
Name: count, dtype: int64

--- 'SUB_COMMODITY_DESC' 고유값 2383개, 상위 10개 ---
SUB_COMMODITY_DESC
CARDS EVERYDAY            1005
BEERALEMALT LIQUORS        833
SPICES & SEASONINGS   

C:\Users\spide\AppData\Local\Temp\ipykernel_21052\2817218715.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns


## 2. hh_demographic — household_key 기준 전체 커버리지

전체 안정구간 가구 대비 인구통계 보유 비율. (지난번 8개 테이블 개괄에서 본 32%와 일치하는지 재확인)


In [24]:
tx = pd.read_csv(DATA_DIR + "transaction_data.csv")

STABLE_MIN_WEEK = 17
STABLE_MAX_WEEK = 99
EARLY_N = 10

tx_stable = tx[(tx["WEEK_NO"] >= STABLE_MIN_WEEK) & (tx["WEEK_NO"] <= STABLE_MAX_WEEK)].copy()
stable_households = tx_stable["household_key"].unique()

demo_hh = set(dfs["hh_demographic"]["household_key"].unique())
stable_hh_set = set(stable_households)

coverage = len(stable_hh_set & demo_hh) / len(stable_hh_set) * 100
print(f"안정구간 전체 가구 중 hh_demographic 보유 비율: {coverage:.1f}%")


안정구간 전체 가구 중 hh_demographic 보유 비율: 32.1%


## 3. hh_demographic — 초반 상위20% 가구(499명)만 놓고 커버리지 재확인

모델 학습 대상이 되는 상위20% 가구에서는 전체 평균(32%)과 다를 수 있어 별도로 확인.


In [25]:
early_window = tx_stable[
    (tx_stable["WEEK_NO"] >= STABLE_MIN_WEEK) &
    (tx_stable["WEEK_NO"] < STABLE_MIN_WEEK + EARLY_N)
]
sales_early_full = early_window.groupby("household_key")["SALES_VALUE"].sum()

panel_wide = pd.DataFrame(index=stable_households)
panel_wide["sales_early"] = sales_early_full.reindex(panel_wide.index).fillna(0)

cutoff_value = panel_wide["sales_early"].quantile(0.8)
panel_wide["group"] = np.where(panel_wide["sales_early"] >= cutoff_value, "상위20%", "나머지80%")

top20_households = set(panel_wide[panel_wide["group"] == "상위20%"].index)
print(f"초반 기준 상위20% 가구 수: {len(top20_households):,}")

top20_coverage = len(top20_households & demo_hh) / len(top20_households) * 100
print(f"상위20% 가구 중 hh_demographic 보유 비율: {top20_coverage:.1f}%")
print(f"→ 결측(정보없음)으로 처리될 가구 수: {len(top20_households) - len(top20_households & demo_hh)}")


초반 기준 상위20% 가구 수: 499
상위20% 가구 중 hh_demographic 보유 비율: 63.9%
→ 결측(정보없음)으로 처리될 가구 수: 180


## 4. causal_data — 초반 구간(week 17~26)만 필터링했을 때 규모

전체 규모와, 실제 feature 계산에 쓸 초반 구간만 남겼을 때 규모를 같이 확인 (누수 방지 확인 겸).


In [26]:
print(dfs["causal_data"].columns.tolist())

['PRODUCT_ID', 'STORE_ID', 'WEEK_NO', 'display', 'mailer']


In [27]:
causal_early = dfs["causal_data"][
    (dfs["causal_data"]["WEEK_NO"] >= STABLE_MIN_WEEK) &
    (dfs["causal_data"]["WEEK_NO"] < STABLE_MIN_WEEK + EARLY_N)
]

print(f"causal_data 전체 행 수: {len(dfs['causal_data']):,}")
print(f"초반 구간(week 17~26)만 필터링 후: {len(causal_early):,} ({len(causal_early)/len(dfs['causal_data'])*100:.1f}%)")

print("\nDISPLAY 값 분포 (초반 구간)")
print(causal_early["display"].value_counts())

print("\nMAILER 값 분포 (초반 구간)")
print(causal_early["mailer"].value_counts())


causal_data 전체 행 수: 36,786,524
초반 구간(week 17~26)만 필터링 후: 3,844,057 (10.4%)

DISPLAY 값 분포 (초반 구간)
display
0    2095870
9     293258
5     290242
7     251627
3     230012
6     208397
2     203205
1     123593
A      80146
4      67707
Name: count, dtype: int64

MAILER 값 분포 (초반 구간)
mailer
A    1814795
0    1288246
D     372445
H     189788
F      93230
L      43977
C      22659
J      18917
Name: count, dtype: int64


## 5. coupon_redempt — 초반 구간(DAY 기준)만 필터링했을 때 규모

coupon_redempt는 WEEK_NO가 없고 DAY 컬럼만 있으므로, week→day 변환해서 필터링.


In [28]:
early_day_min = tx.loc[tx["WEEK_NO"] == STABLE_MIN_WEEK, "DAY"].min()
early_day_max = tx.loc[tx["WEEK_NO"] == STABLE_MIN_WEEK + EARLY_N - 1, "DAY"].max()
print(f"초반 구간 DAY 범위: {early_day_min} ~ {early_day_max}")

coupon_redempt_early = dfs["coupon_redempt"][
    (dfs["coupon_redempt"]["DAY"] >= early_day_min) &
    (dfs["coupon_redempt"]["DAY"] <= early_day_max)
]

print(f"\ncoupon_redempt 전체 행 수: {len(dfs['coupon_redempt']):,}")
print(f"초반 구간만 필터링 후: {len(coupon_redempt_early):,} ({len(coupon_redempt_early)/len(dfs['coupon_redempt'])*100:.1f}%)")

# 상위20% 가구 중 초반 구간에 쿠폰 상환 이력이 있는 가구 비율
redeemers_early = set(coupon_redempt_early["household_key"].unique())
top20_redeemer_pct = len(top20_households & redeemers_early) / len(top20_households) * 100
print(f"\n상위20% 가구 중 초반 구간 쿠폰 상환 이력 있는 비율: {top20_redeemer_pct:.1f}%")
print("→ 이 비율이 낮으면(희소 feature) 모델에서 이진(0/1) 형태로 쓰는 게 안전")


초반 구간 DAY 범위: 111 ~ 180

coupon_redempt 전체 행 수: 2,318
초반 구간만 필터링 후: 0 (0.0%)

상위20% 가구 중 초반 구간 쿠폰 상환 이력 있는 비율: 0.0%
→ 이 비율이 낮으면(희소 feature) 모델에서 이진(0/1) 형태로 쓰는 게 안전


## 6. product — causal_data / transaction_data와 조인 키 확인

product_id 중복 여부, 결측 확인 (조인 시 다대다 매칭 등 문제 없는지).


In [29]:
print(f"product_id 고유값 수: {dfs['product']['PRODUCT_ID'].nunique():,}")
print(f"전체 행 수: {len(dfs['product']):,}")
print(f"product_id 중복 여부: {'있음 - 확인 필요' if dfs['product']['PRODUCT_ID'].nunique() != len(dfs['product']) else '없음 (1:1 매핑 확인됨)'}")


product_id 고유값 수: 92,353
전체 행 수: 92,353
product_id 중복 여부: 없음 (1:1 매핑 확인됨)


## 7. 다음 단계 메모

- hh_demographic 결측은 "정보없음" 카테고리로 포함 (3번 셀 결과 참고해서 실제 결측 규모 확인)
- causal_data, coupon_redempt는 반드시 초반 구간(week 17~26 / 해당 DAY 범위)만 필터링해서 feature 계산 — 4, 5번 셀의 필터링 로직 그대로 재사용
- coupon_redempt 상환 이력 비율이 낮다면(희소), "상환 횟수"보다 "상환 여부(0/1)"로 단순화하는 것을 고려


## 8. campaign_table / campaign_desc — 초반 구간에 캠페인이 있었는지 확인

coupon_redempt가 초반 구간(DAY 111~180)에 전혀 없었던 것과 같은 문제가 있는지,
캠페인 발송 자체는 이 구간에 있었는지 확인한다.


In [30]:
dfs["campaign_table"] = pd.read_csv(DATA_DIR + "campaign_table.csv")
dfs["campaign_desc"] = pd.read_csv(DATA_DIR + "campaign_desc.csv")

for name in ["campaign_table", "campaign_desc"]:
    df = dfs[name]
    print(f"===== {name} (shape: {df.shape}) =====")
    display(df.head(3))
    print("dtypes:")
    print(df.dtypes)
    n_dup = df.duplicated().sum()
    print(f"완전 중복행: {n_dup:,}")
    na_ratio = (df.isna().sum() / len(df) * 100).round(2)
    print("결측치 비율(%):")
    print(na_ratio[na_ratio > 0] if na_ratio.sum() > 0 else "결측 없음")
    print()


===== campaign_table (shape: (7208, 3)) =====


,DESCRIPTION,household_key,CAMPAIGN
0,TypeA,17,26
1,TypeA,27,26
2,TypeA,212,26


dtypes:
DESCRIPTION        str
household_key    int64
CAMPAIGN         int64
dtype: object
완전 중복행: 0
결측치 비율(%):
결측 없음

===== campaign_desc (shape: (30, 4)) =====


,DESCRIPTION,CAMPAIGN,START_DAY,END_DAY
0,TypeB,24,659,719
1,TypeC,15,547,708
2,TypeB,25,659,691


dtypes:
DESCRIPTION      str
CAMPAIGN       int64
START_DAY      int64
END_DAY        int64
dtype: object
완전 중복행: 0
결측치 비율(%):
결측 없음



### 8-1. 캠페인 시작일(START_DAY) 분포 — 초반 구간(DAY 111~180)과 겹치는지

In [31]:
print("campaign_desc START_DAY / END_DAY 요약")
print(dfs["campaign_desc"][["START_DAY", "END_DAY"]].describe())

early_day_min = 111  # 앞서 확인한 초반 구간 DAY 범위 재사용
early_day_max = 180

# 캠페인 기간이 초반 구간과 조금이라도 겹치는 경우 (시작일이 초반구간 끝나기 전, 종료일이 초반구간 시작 이후)
overlapping_campaigns = dfs["campaign_desc"][
    (dfs["campaign_desc"]["START_DAY"] <= early_day_max) &
    (dfs["campaign_desc"]["END_DAY"] >= early_day_min)
]

print(f"\n초반 구간(DAY {early_day_min}~{early_day_max})과 겹치는 캠페인 수: {len(overlapping_campaigns)} / 전체 {len(dfs['campaign_desc'])}")
display(overlapping_campaigns)


campaign_desc START_DAY / END_DAY 요약
        START_DAY     END_DAY
count   30.000000   30.000000
mean   463.866667  510.466667
std    134.488490  137.730555
min    224.000000  264.000000
25%    360.000000  405.750000
50%    470.000000  502.000000
75%    584.000000  640.250000
max    659.000000  719.000000

초반 구간(DAY 111~180)과 겹치는 캠페인 수: 0 / 전체 30


,DESCRIPTION,CAMPAIGN,START_DAY,END_DAY


### 8-2. 상위20% 가구 중, 초반 구간과 겹치는 캠페인을 받은 비율

In [32]:
if len(overlapping_campaigns) > 0:
    overlapping_campaign_ids = overlapping_campaigns["CAMPAIGN"].unique()

    campaign_table_early = dfs["campaign_table"][
        dfs["campaign_table"]["CAMPAIGN"].isin(overlapping_campaign_ids)
    ]

    print(f"초반 구간 겹치는 캠페인의 발송 기록 수: {len(campaign_table_early):,}")

    recipients_early = set(campaign_table_early["household_key"].unique())
    top20_campaign_pct = len(top20_households & recipients_early) / len(top20_households) * 100

    print(f"상위20% 가구 중 초반 구간 캠페인 수신 이력 있는 비율: {top20_campaign_pct:.1f}%")

    all_recipients_pct = len(stable_hh_set & recipients_early) / len(stable_hh_set) * 100
    print(f"안정구간 전체 가구 중 초반 구간 캠페인 수신 이력 있는 비율: {all_recipients_pct:.1f}%")
else:
    print("초반 구간과 겹치는 캠페인이 없음 → 이 feature도 사용 불가, coupon_redempt와 동일하게 제외 검토")


초반 구간과 겹치는 캠페인이 없음 → 이 feature도 사용 불가, coupon_redempt와 동일하게 제외 검토


### 8-3. 판단

- 8-1에서 겹치는 캠페인이 0개면 → campaign 관련 feature도 coupon_redempt와 같은 이유로 초반 구간에서는 사용 불가
- 겹치는 캠페인이 있고, 8-2의 두 비율(상위20% vs 전체)이 차이를 보인다면 → "초반 구간 캠페인 수신 여부(0/1)"를 feature로 채택 가능
- 캠페인 타입(Type A/B/C)별로 나눠서 더 세분화할 수도 있으나, 표본 크기(499명)를 고려해 우선 수신 여부(이진)로 단순화 권장
